# H2: Theoretical vs. Empirical Bounds

**Hypothesis H2:** The empirical number of private accesses to realize k hits stays below the
prepaid theoretical bound K_k; the empirical failure rate is ≤ δ.

**What K_k bounds:**  
K_k = (1 + α_k) / q · k is the (1−δ_k)-quantile of NegBinom(k, q), where q = 1/(e^ε + 1)
is the per-round hit probability when the true query sits exactly at threshold (worst case /
tightest case). The Zeta-2 allocation δ_k = 6/(π²k²)·δ ensures the bound holds simultaneously
for all k ≥ 1 (anytime guarantee), with union-bound probability ≤ δ total.

**Why simulate with p_hit = q:**  
When the true query equals T, P(Lap(1/ε) ≥ 0) = q exactly — this is the adversarial
case that maximises the number of rounds needed per hit. In the attribution deployment,
p_hit = TARGET_HIT_RATE ≪ q, so the bound holds more conservatively. Testing with p_hit = q
isolates the mathematical claim and demonstrates tightness.

**Failure event:** ∃ k ≥ 1 such that empirical_rounds_to_k_hits > K_k.  
P(failure) should be ≤ δ.

**Sections:**
1. K_k curves and single-run illustration
2. Theoretical vs. empirical: percentile bands over N_RUNS independent runs
3. Empirical failure rate vs. δ

In [ ]:
import sys
sys.path.insert(0, '..')

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats as scipy_stats

from simulation.accountants.adaptive_tct import AdaptiveTCTAccountant
from simulation.calibration import q_from_epsilon

plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'legend.fontsize': 9,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'lines.linewidth': 2.0,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

TCT_COLOR  = '#2166ac'
BAND_COLOR = '#92c5de'
FAIL_COLOR = '#d6604d'

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
EPSILON    = 0.01       # ε for Laplace mechanism; determines q
DELTA_MAIN = 1e-2       # primary δ for main H2 plots
MAX_HITS   = 30         # track K_k and empirical rounds for k = 1 … MAX_HITS
N_RUNS     = 5000       # independent runs for empirical distributions
SEED       = 42

# δ values for failure-rate sweep — moderate range so observed failures are
# statistically significant (expected failures = N_RUNS × δ).
DELTA_SWEEP = [0.20, 0.10, 0.05, 0.02, 0.01, 0.005]

q = q_from_epsilon(EPSILON)
hit_ks = list(range(1, MAX_HITS + 1))

acc_main = AdaptiveTCTAccountant(EPSILON, DELTA_MAIN)
K_bounds  = np.array([acc_main.compositions(k) for k in hit_ks])

print(f'ε={EPSILON},  q={q:.4f}')
print(f'E[rounds per hit] = 1/q = {1/q:.2f}')
print()
header = f"{'k':>4}  {'a_k':>8}  {'K_k':>10}  {'K_k/E':>8}  {'d_k':>12}"
print(header)
print('-' * 50)
for k in [1, 2, 5, 10, 20, 30]:
    alpha_k = acc_main.alpha(k)
    K_k     = acc_main.compositions(k)
    delta_k = (6 / (math.pi**2 * k**2)) * DELTA_MAIN
    print(f'{k:>4}  {alpha_k:>8.3f}  {K_k:>10.2f}  {K_k/(k/q):>8.3f}  {delta_k:>12.2e}')

In [ ]:
def simulate_hit_rounds(n_runs: int, max_hits: int, hit_prob: float,
                        seed: int = SEED) -> np.ndarray:
    """
    Simulate n_runs independent trials.  In each trial, rounds proceed
    until max_hits tops are accumulated; each round is a hit independently
    with probability hit_prob.

    Returns array of shape (n_runs, max_hits) where entry [i, k-1] is
    the round number of the k-th hit in run i.
    """
    rng = np.random.default_rng(seed)
    # Geometric(p): rounds until next hit.  Sum of k geometrics = NegBinom(k, p).
    waiting = rng.geometric(p=hit_prob, size=(n_runs, max_hits))
    return np.cumsum(waiting, axis=1)  # (n_runs, max_hits)


def failure_rate_ci(failures: np.ndarray) -> tuple[float, float, float]:
    """Wilson 95% CI for a binary success/failure array."""
    n = len(failures)
    p = failures.mean()
    z = 1.96
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half   = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return float(p), float(centre - half), float(centre + half)


# Main simulation: p_hit = q (tight/adversarial case)
print('Running main simulation (p_hit = q)…')
hit_rounds_main = simulate_hit_rounds(N_RUNS, MAX_HITS, hit_prob=q)
print(f'  Shape: {hit_rounds_main.shape}')
print(f'  Median rounds to 1st hit: {np.median(hit_rounds_main[:, 0]):.1f}  '
      f'(E = {1/q:.1f})')
print(f'  Median rounds to {MAX_HITS}th hit: {np.median(hit_rounds_main[:, -1]):.1f}  '
      f'(E = {MAX_HITS/q:.1f})')

## Section 1: K_k Curves and Tightness

In [ ]:
# Percentile bands over N_RUNS runs
pct_05  = np.percentile(hit_rounds_main, 5,  axis=0)
pct_25  = np.percentile(hit_rounds_main, 25, axis=0)
pct_50  = np.percentile(hit_rounds_main, 50, axis=0)
pct_75  = np.percentile(hit_rounds_main, 75, axis=0)
pct_95  = np.percentile(hit_rounds_main, 95, axis=0)
pct_99  = np.percentile(hit_rounds_main, 99, axis=0)
expected = np.array([k / q for k in hit_ks])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# ── Left: K_k vs empirical rounds ────────────────────────────────────────────
ax = axes[0]
ax.fill_between(hit_ks, pct_05, pct_95, alpha=0.15, color=TCT_COLOR,
                label='5th–95th pct (empirical)')
ax.fill_between(hit_ks, pct_25, pct_75, alpha=0.25, color=TCT_COLOR,
                label='IQR (empirical)')
ax.plot(hit_ks, pct_50,   color=TCT_COLOR, lw=2,   label='Median (empirical)')
ax.plot(hit_ks, K_bounds, color='black',   lw=2,   linestyle='--',
        label=f'$K_k$ bound  (δ={DELTA_MAIN:.0e})')
ax.plot(hit_ks, expected, color='#888888', lw=1.2, linestyle=':',
        label='$k/q$ (expected rounds)')
ax.set_xlabel('Hit number $k$')
ax.set_ylabel('Rounds')
ax.set_title('Theoretical Bound $K_k$ vs. Empirical Rounds')
ax.set_xlim(1, MAX_HITS)
ax.set_ylim(bottom=0)
ax.legend(fontsize=8)

# ── Right: Overhead ratio K_k / (k/q) ────────────────────────────────────────
ax = axes[1]
overhead = K_bounds / expected          # (1 + α_k)
emp_ratio_99 = pct_99 / expected        # 99th pct / mean
emp_ratio_50 = pct_50 / expected

ax.plot(hit_ks, overhead,     color='black',   lw=2,   linestyle='--',
        label='Bound overhead $(1+\\alpha_k)$')
ax.plot(hit_ks, emp_ratio_99, color=FAIL_COLOR, lw=1.5, linestyle='-',
        label='99th pct / mean (empirical)')
ax.plot(hit_ks, emp_ratio_50, color=TCT_COLOR,  lw=1.5, linestyle='-',
        label='Median / mean (empirical)')
ax.axhline(1.0, color='#888888', lw=0.8, linestyle=':')
ax.set_xlabel('Hit number $k$')
ax.set_ylabel('Ratio to $k/q$')
ax.set_title('Overhead Factor $(1+\\alpha_k)$')
ax.set_xlim(1, MAX_HITS)
ax.set_ylim(bottom=0)
ax.legend(fontsize=8)

fig.suptitle(
    f'H2: Theoretical $K_k$ vs. Empirical Rounds — '
    f'ε={EPSILON}, δ={DELTA_MAIN:.0e}, p_hit=q={q:.4f}, N={N_RUNS} runs',
    fontsize=11,
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('../figures/h2_bounds_main.png', bbox_inches='tight', dpi=200)
plt.show()

# Spot-check: fraction of runs that exceed K_k for each k
exceed_frac = np.mean(hit_rounds_main > K_bounds[np.newaxis, :], axis=0)
print('Fraction of runs exceeding K_k per hit:')
for k in [1, 5, 10, 20, 30]:
    print(f'  k={k:2d}:  {exceed_frac[k-1]:.4f}  (δ_k = {(6/(math.pi**2*k**2))*DELTA_MAIN:.4e},  K_k={K_bounds[k-1]:.1f})')

## Section 2: Empirical Failure Rate vs. δ

A **failure** in run i is: ∃ k ∈ {1, …, MAX_HITS} such that empirical_rounds[i, k] > K_k(δ).
The guarantee (Theorem 3 + union bound) is P(failure) ≤ δ.

We sweep δ ∈ {0.20, 0.10, 0.05, 0.02, 0.01, 0.005} and estimate the empirical failure rate
from N_RUNS independent runs at each δ. The same random hit sequences are reused;
only the K_k bounds change with δ.

In [ ]:
rows = []
for delta in DELTA_SWEEP:
    acc = AdaptiveTCTAccountant(EPSILON, delta)
    bounds_d = np.array([acc.compositions(k) for k in hit_ks])
    # failure: any k in 1..MAX_HITS where empirical rounds > K_k
    failures = np.any(hit_rounds_main > bounds_d[np.newaxis, :], axis=1)
    rate, lo, hi = failure_rate_ci(failures)
    rows.append({
        'delta': delta,
        'rate': rate, 'lo': lo, 'hi': hi,
        'n_fail': int(failures.sum()),
        'K1': acc.compositions(1),
    })
    print(f'δ={delta:.3f}  K₁={acc.compositions(1):6.1f}  '
          f'failures={int(failures.sum()):4d}/{N_RUNS}  '
          f'rate={rate:.4f}  95%CI=[{lo:.4f},{hi:.4f}]  bound_ok={rate<=delta}')

df_fr = pd.DataFrame(rows)

In [ ]:
# ── Figure: Failure rate vs. δ ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))

deltas = df_fr['delta'].values
rates  = df_fr['rate'].values
errs   = np.array([
    rates - df_fr['lo'].values,
    df_fr['hi'].values - rates,
])

ax.plot([1e-3, 0.3], [1e-3, 0.3], color='#888888', lw=1.2, linestyle='--',
        label='y = δ  (exact bound)')
ax.errorbar(deltas, rates, yerr=errs, fmt='o', color=FAIL_COLOR,
            capsize=4, capthick=1.5, ms=7, lw=1.8,
            label=f'Empirical failure rate ± 95% CI  (N={N_RUNS})')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('δ (theoretical bound)')
ax.set_ylabel('Empirical failure rate')
ax.set_title('Empirical Failure Rate vs. δ')
ax.set_xlim(3e-3, 0.35)
ax.set_ylim(3e-3, 0.35)
ax.legend(fontsize=9)

# Annotate whether each point is below the bound
for _, row in df_fr.iterrows():
    marker = '✓' if row['rate'] <= row['delta'] else '✗'
    ax.annotate(marker,
                xy=(row['delta'], row['rate']),
                xytext=(4, 4), textcoords='offset points',
                fontsize=10, color='green' if row['rate'] <= row['delta'] else 'red')

fig.suptitle(
    f'H2: Failure Rate — ε={EPSILON}, δ swept {DELTA_SWEEP[0]}–{DELTA_SWEEP[-1]}, p_hit=q={q:.4f}, '
    f'k=1…{MAX_HITS}, N={N_RUNS}',
    fontsize=11,
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('../figures/h2_failure_rate.png', bbox_inches='tight', dpi=200)
plt.show()

## Section 3: Tightness — K_k vs. High Quantiles of Empirical Distribution

For the bound to be non-vacuous, K_k should track the (1−δ)-quantile of the empirical
rounds-to-k-hits distribution.  We compare K_k to the empirical (1−δ_k)-quantile where
δ_k = 6/(π²k²)·δ is the per-hit allocation used by the Zeta-2 scheme.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

acc_main = AdaptiveTCTAccountant(EPSILON, DELTA_MAIN)

emp_quantiles = []
for k in hit_ks:
    delta_k = (6 / (math.pi**2 * k**2)) * DELTA_MAIN
    q_level = 1.0 - delta_k
    emp_q   = np.quantile(hit_rounds_main[:, k-1], q_level)
    emp_quantiles.append(emp_q)
emp_quantiles = np.array(emp_quantiles)

ax.plot(hit_ks, K_bounds,      color='black',    lw=2,   linestyle='--',
        label=f'$K_k$ (theoretical bound, δ={DELTA_MAIN:.0e})')
ax.plot(hit_ks, emp_quantiles, color=TCT_COLOR,  lw=2,   linestyle='-',
        label=f'$(1-\\delta_k)$-quantile of empirical distribution')
ax.fill_between(hit_ks, emp_quantiles, K_bounds,
                where=K_bounds >= emp_quantiles,
                alpha=0.12, color=TCT_COLOR, label='Bound gap (K_k − empirical)')

ax.set_xlabel('Hit number $k$')
ax.set_ylabel('Rounds')
ax.set_title('$K_k$ vs. Matching Empirical Quantile')
ax.set_xlim(1, MAX_HITS)
ax.set_ylim(bottom=0)
ax.legend(fontsize=9)

fig.suptitle(
    f'H2 Tightness — ε={EPSILON}, δ={DELTA_MAIN:.0e}, '
    f'p_hit=q={q:.4f}, N={N_RUNS}',
    fontsize=11,
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('../figures/h2_tightness.png', bbox_inches='tight', dpi=200)
plt.show()

# Ratio table
print(f"{'k':>4}  {'K_k':>8}  {'Emp q':>8}  {'Ratio':>7}  {'d_k':>10}")
print('-' * 45)
for k in [1, 2, 5, 10, 20, 30]:
    dk = (6 / (math.pi**2 * k**2)) * DELTA_MAIN
    eq = np.quantile(hit_rounds_main[:, k-1], 1 - dk)
    Kk = acc_main.compositions(k)
    print(f'{k:>4}  {Kk:>8.1f}  {eq:>8.1f}  {Kk/eq:>7.3f}  {dk:>10.2e}')

## Section 4: Summary

In [ ]:
print('H2 Summary')
print('=' * 60)
print(f'ε={EPSILON}, q={q:.4f}, MAX_HITS={MAX_HITS}, N_RUNS={N_RUNS}')
print()
print('Failure rate results (all runs, k=1…MAX_HITS):')
print(f"{'delta':>7}  {'K1':>7}  {'failures':>9}  {'rate':>7}  {'<=d?':>5}")
print('-' * 45)
for _, row in df_fr.iterrows():
    ok = 'ok' if row['rate'] <= row['delta'] else 'FAIL'
    print(f"{row['delta']:>7.3f}  {row['K1']:>7.1f}  "
          f"{int(row['n_fail']):>9d}  {row['rate']:>7.4f}  {ok:>5}")

print()
print('Overhead factor at key hit counts (K_k / E[rounds]):')
acc_main = AdaptiveTCTAccountant(EPSILON, DELTA_MAIN)
for k in [1, 5, 10, 20, 30]:
    Kk = acc_main.compositions(k)
    overhead = Kk / (k / q)
    print(f'  k={k:2d}:  K_k={Kk:7.1f}  E={k/q:6.1f}  overhead={overhead:.3f}')

## Paper Figure: Combined H2 Bound Validity (Figure 3)

In [ ]:
# ── Paper Figure 3: H2 combined — tightness + failure rate (publication quality)
# Requires: hit_ks, K_bounds, emp_quantiles (from cell-10)
#           hit_rounds_main (from cell-03), df_fr (from cell-07)
#           DELTA_MAIN, DELTA_SWEEP, N_RUNS, MAX_HITS, q, EPSILON

fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))

# ── Panel (a): K_k vs empirical (1−δ_k)-quantile ─────────────────────────────
ax = axes[0]
ax.plot(hit_ks, K_bounds,      color='black',    lw=2.0, linestyle='--',
        label=f'$K_k$ bound  ($\\delta$={DELTA_MAIN:.0e})')
ax.plot(hit_ks, emp_quantiles, color=TCT_COLOR,  lw=2.0,
        label='$(1-\\delta_k)$-quantile (empirical)')
ax.fill_between(hit_ks, emp_quantiles, K_bounds,
                where=np.array(K_bounds) >= np.array(emp_quantiles),
                alpha=0.15, color=TCT_COLOR, label='Bound gap ($K_k$ − quantile)')
ax.set_xlabel('Hit number $k$')
ax.set_ylabel('Rounds to $k$ hits')
ax.set_title('(a) $K_k$ vs. Matching Empirical Quantile', fontsize=10)
ax.set_xlim(1, MAX_HITS)
ax.set_ylim(bottom=0)
ax.legend(fontsize=8.5)

# ── Panel (b): Anytime failure rate vs δ — all points visible ────────────────
ax = axes[1]

# Use min_rate floor so that zero-failure points appear as ▽ on log scale
min_rate = 0.5 / N_RUNS

deltas_main = df_fr['delta'].values
rates_main  = df_fr['rate'].values
lo_main     = df_fr['lo'].values
hi_main     = df_fr['hi'].values

has_fail = rates_main > 0
no_fail  = ~has_fail

# y=δ reference line
diag = np.array([min_rate * 0.3, 0.5])
ax.plot(diag, diag, color='#888', lw=1.2, linestyle='--', zorder=1,
        label='$y = \\delta$  (exact bound)')

# Points with observed failures: circles with error bars
if has_fail.any():
    errs_nz = np.array([
        rates_main[has_fail] - lo_main[has_fail],
        hi_main[has_fail] - rates_main[has_fail],
    ])
    ax.errorbar(deltas_main[has_fail], rates_main[has_fail],
                yerr=errs_nz, fmt='o', color=FAIL_COLOR,
                capsize=4, capthick=1.5, ms=7, lw=1.8, zorder=3,
                label=f'Empirical rate ± 95% CI  ($N$={N_RUNS})')

# Points with zero failures: downward triangles at floor, labeled separately
if no_fail.any():
    ax.scatter(deltas_main[no_fail],
               np.full(no_fail.sum(), min_rate * 1.8),
               marker='v', s=60, color=FAIL_COLOR, alpha=0.75, zorder=3,
               label=f'0/{N_RUNS} failures  (below axis)')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('$\\delta$ (failure budget)')
ax.set_ylabel('Empirical anytime failure rate')
ax.set_title('(b) Anytime Failure Rate vs. $\\delta$', fontsize=10)
ax.set_xlim(min(deltas_main) * 0.3, max(deltas_main) * 3)
ax.set_ylim(min_rate * 0.3, max(deltas_main) * 3)
ax.legend(fontsize=8.5, loc='upper left')

fig.suptitle(
    rf'Bound Validity and Tightness — $\varepsilon$={EPSILON}, $\delta$={DELTA_MAIN:.0e}, '
    rf'$p_{{\mathrm{{hit}}}}=q={q:.4f}$, $k=1\ldots{MAX_HITS}$, '
    rf'$N_{{\mathrm{{runs}}}}$={N_RUNS}',
    fontsize=10,
)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig('../figures/paper_fig_h2.png', bbox_inches='tight', dpi=200)
plt.show()

# Summary table
print('Paper Figure 3 — H2 summary')
print(f"{'delta':>7}  {'K1':>7}  {'failures':>9}  {'rate':>8}  {'≤δ?':>5}")
print('-' * 45)
for _, row in df_fr.iterrows():
    ok = 'ok' if row['rate'] <= row['delta'] else 'FAIL'
    print(f"{row['delta']:>7.3f}  {row['K1']:>7.1f}  "
          f"{int(row['n_fail']):>9d}  {row['rate']:>8.4f}  {ok:>5}")
print()
print('Tightness ratio K_k / empirical (1−δ_k)-quantile:')
for k in [1, 5, 10, 20, 30]:
    ratio = float(K_bounds[k-1]) / float(emp_quantiles[k-1])
    print(f'  k={k:2d}: K_k={K_bounds[k-1]:.1f},  quantile={emp_quantiles[k-1]:.1f},  '
          f'ratio={ratio:.2f}×')